In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
import torch
import geopandas as gpd
import pandas as pd
import os
from pathlib import Path
import sys
def configure_proj_data():
    candidates = [
        Path(sys.prefix) / "Library" / "share" / "proj",
        Path(sys.prefix) / "Lib" / "site-packages" / "pyproj" / "proj_dir" / "share" / "proj",
        Path(os.environ.get("CONDA_PREFIX", "")) / "Library" / "share" / "proj",
        Path.home() / ".conda" / "envs" / "3s" / "Library" / "share" / "proj",
    ]

    for candidate in candidates:
        if (candidate / "proj.db").exists():
            os.environ["PROJ_LIB"] = str(candidate)
            os.environ["PROJ_DATA"] = str(candidate)

            import pyproj
            pyproj.datadir.set_data_dir(str(candidate))

            print("PROJ database:", candidate)
            return

    print("WARNING: proj.db not found")

configure_proj_data()

In [ ]:
dem_file = gpd.read_file(r"GIS_features/dem_features_15msa.csv")
print(dem_file)
ndvi_file = gpd.read_file(r"GIS_features/ndvi_features_15msa.csv")
print(ndvi_file)
nlcd_file = gpd.read_file(r"GIS_features/nlcd_features_15msa.csv")
print(nlcd_file)
poi_file = gpd.read_file(r"GIS_features/poi_features_15msa.csv")
print(poi_file)
pop_file = gpd.read_file(r"GIS_features/pop_features_15msa.csv")
print(pop_file)
eco_file = gpd.read_file(r"GIS_features/socioeconomic_features_15msa.csv")
print(eco_file)
tcc_file = gpd.read_file(r"GIS_features/tcc_features_15msa.csv")
print(tcc_file)
water_file = gpd.read_file(r"GIS_features/water_dist_features_15msa.csv")
print(water_file)

In [ ]:
gdf = dem_file.merge(ndvi_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(nlcd_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(poi_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(pop_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(eco_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(tcc_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(water_file,
                on="cb_2020_3",
                how="inner")
print(gdf)

In [ ]:
print(gdf.columns)

In [ ]:
import numpy as np
feature_cols = ['elev_mean', 'slope_mean', 'ndvi_mean', 'water_pct',
       'developed_pct', 'barren_pct', 'forest_pct',
       'grassland_pct', 'pasture_hay_pct', 'cultivated_crops_pct',
       'woody_wetlands_pct', 'herbaceous_wetlands_pct', 'impervious_mean',
       'poi_edu', 'poi_comm', 'poi_med', 'poi_trans', 'poi_pub', 'poi_food',
       'poi_finance', 'poi_total_all', 'poi_diversity', 'minor_ratio',
       'elderly_ratio', 'sex_ratio', 'fam_inc_lt10k', 'fam_inc_lt10k_moe',
       'fam_inc_10to15k', 'fam_inc_10to15k_moe', 'fam_inc_15to25k',
       'fam_inc_15to25k_moe', 'fam_inc_25to50k', 'fam_inc_25to50k_moe',
       'fam_inc_50kplus', 'fam_inc_50kplus_moe', 'median_family_income',
       'median_family_income_moe', 'per_capita_income',
       'per_capita_income_moe', 'persons_below_poverty',
       'persons_below_poverty_moe', 'tcc_mean', 'dist_water_m']
  
gdf_corr = gdf[feature_cols].copy()

for col in feature_cols:
    gdf_corr[col] = pd.to_numeric(
        gdf_corr[col],
        errors="coerce"
    )
gdf_corr = gdf_corr.dropna()
corr_matrix = gdf_corr.corr(method="pearson")
print(corr_matrix)

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr = upper.stack()
high_corr = high_corr[abs(high_corr) > 0.6]

print("相关性绝对值 > 0.8 的变量：")
print(high_corr)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(18,15))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True
)

plt.tight_layout()
plt.show()

In [ ]:
feature_cols = ['elev_mean', 'slope_mean', 'ndvi_mean', 'water_pct',
       'developed_pct', 'barren_pct', 'forest_pct',
       'grassland_pct', 'pasture_hay_pct', 'cultivated_crops_pct',
       'woody_wetlands_pct', 'herbaceous_wetlands_pct',
       'poi_edu', 'poi_comm', 'poi_med', 'poi_trans', 'poi_pub', 'poi_food',
       'poi_finance', 'poi_diversity', 'minor_ratio',
       'elderly_ratio', 'sex_ratio', 'fam_inc_lt10k', 
       'fam_inc_10to15k', 'fam_inc_15to25k',
       'fam_inc_25to50k', 
       'fam_inc_50kplus',  'median_family_income',
       'persons_below_poverty', 'dist_water_m']
  
gdf_corr = gdf[feature_cols].copy()

for col in feature_cols:
    gdf_corr[col] = pd.to_numeric(
        gdf_corr[col],
        errors="coerce"
    )
gdf_corr = gdf_corr.dropna()
corr_matrix = gdf_corr.corr(method="pearson")
print(corr_matrix)


In [ ]:
plt.figure(figsize=(18,15))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True
)

plt.tight_layout()
plt.show()